In [1]:
# Imports
import sys
import logging
from datetime import datetime
import pandas as pd
from IPython.display import display

sys.path.insert(0, '../../../LOGOS')
from src import Pert, plot_gantt_chart, plot_resource_utilization, plot_location_utilization, plot_equipment_utilization
# Configure logging in the runner (avoid setting basicConfig inside the module)
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
import json
from pathlib import Path

cwd = Path.cwd()
benchmark = cwd/'benchmarks'
benchmark_results_file = benchmark/'priority_rules_results.json'

## Load Benchmark results
with open(benchmark_results_file, "r", encoding="utf-8") as f:
    benchmark_data = json.load(f)

In [2]:
def run_case(case_name, file_name, json_path,
             schema_file="outage_schema.json",
             benchmark_data=None):
    """
    Runs scheduling comparison for a given case and file.

    Parameters:
        case_name (str): Case identifier (e.g., 'j60')
        file_name (str): File name (e.g., 'j601_1.sm')
        json_path (str): Path to JSON file for Pert model
        schema_file (str): Path to schema file (default: outage_schema.json)
        benchmark_data (dict): Benchmark dataset for RCPSP comparison

    Returns:
        results_df (pd.DataFrame): LOGOS.CPM results
        data_df (pd.DataFrame): RCPSP benchmark results
    """

    results_sgs = {}
    results_pgs = {}

    # Load Pert Model
    pert = Pert.from_json_file(json_path, schema_path=schema_file)

    prs = [
        'lf', 'ls', 'ef', 'es', 'duration', 'random',
        'mts', 'mtp', 'grpw', 'grd', 'rr', 'avgrr',
        'maxrr', 'minrr','mehh_8000_b','mehh_3375_b',
        'mehh_1000_b','mehh_125_b','gphh_b'
    ]

    # Compute results with Serial
    for rule in prs:
        out = pert.calculateSerialScheduleWithResources(priority_rule=rule)
        results_sgs[rule] = out['scheduled_duration'] - 2  # remove start/end duration

    sgs = ['first', 'max_use_res_ranked', 'max_use_res_shuffled', 'md_knapsack', 'look_ahead']

    # Compute results with Parallel
    for s in sgs:
        out = pert.calculateScheduleWithResources(sgs=s)
        results_pgs[s] = out['scheduled_duration'] - 2  # remove start/end duration


    print('Results from LOGOS.CPM Using Serial Generation Scheme:')
    print('-' * 60)
    results_df_sgs = pd.DataFrame(results_sgs, index=[0])
    display(results_df_sgs)

    print('Results from LOGOS.CPM Using Parallel Generation Scheme:')
    print('-' * 60)
    results_df_pgs = pd.DataFrame(results_pgs, index=[0])
    display(results_df_pgs)
    # RCPSP benchmark comparison
    print('Results from RCPSP')
    print('-' * 60)

    if benchmark_data is None:
        raise ValueError("benchmark_data must be provided")

    data = benchmark_data[case_name][file_name]
    data_df = pd.DataFrame(data, index=[0]).filter(like='serial_forward')
    data_df.columns = data_df.columns.str.replace("_serial_forward", "", regex=False)
    data_df.columns = data_df.columns.str.lower()

    display(data_df)

    return results_df_sgs, results_df_pgs, data_df

## Scheduling with 30 activities

In [3]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j30',
    file_name='j301_1.sm',
    json_path='j301_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=32 | CPM=40.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 07:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 05:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J10 | start=2026-01-01 07:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 05:00 | end=2026-01-01 13:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 07:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 14:00 | end=2026-01-01 16:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 09:00 | end=2026-01-01 15:


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J7 | start=2026-01-01 05:00 | end=2026-01-01 10:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J14 | start=2026-01-02 05:00 | end=2026-01-02 08:00 | delay=13.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 09:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 14:00 | end=2026-01-01 16:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 10:00 | end=2026-01-01 12:00 | delay=3.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 12:00 | end=2026-01-01 18:00 | delay=7.0h
DEBUG:root:Serial SGS: scheduled J10 | start=2026-01-01 07:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 05:00 | end=2026-01-01 14:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 07:00 | delay=0.0h
DEBUG:root:Serial SGS: s

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2']
waiting=['J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J2', 'J3', 'J4']
selected=['J2']
t=2026-01-01 05:00
completed=['J1']
ongoing=['J2', 'J3']
waiting=['J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32']
candidates=['J3', 'J4']
selected=['J3']
t=2026-01-01 07:00
completed=['J1']
ongoing=['J2', 'J3', 'J4']
waiting=['J5', 'J6', 'J7',

,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,49.0,46.0,60.0,51.0,44.0,49.0,49.0,44.0,44.0,45.0,49.0,45.0,45.0,49.0,52.0,50.0,52.0,43.0,44.0


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,50.0,43.0,61.0,61.0,43.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,51,60,46,49,57,49,49,61,60,53,52,53,52,46,74



## Scheduling with 60 activities

In [4]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j60',
    file_name='j601_1.sm',
    json_path='j601_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=62 | CPM=79.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 11:00 | end=2026-01-01 20:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J9 | start=2026-01-01 20:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 21:00 | end=2026-01-02 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J14 | start=2026-01-01 02:00 | end=2026-01-01 04:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-02 03:00 | end=2026-01-02 13:


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J46 | start=2026-01-02 23:00 | end=2026-01-03 05:00 | delay=27.0h
DEBUG:root:Serial SGS: scheduled J34 | start=2026-01-01 04:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J36 | start=2026-01-02 14:00 | end=2026-01-02 23:00 | delay=8.0h
DEBUG:root:Serial SGS: scheduled J17 | start=2026-01-01 20:00 | end=2026-01-01 23:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J22 | start=2026-01-02 01:00 | end=2026-01-02 07:00 | delay=10.0h
DEBUG:root:Serial SGS: scheduled J31 | start=2026-01-02 23:00 | end=2026-01-03 02:00 | delay=16.0h
DEBUG:root:Serial SGS: scheduled J29 | start=2026-01-01 02:00 | end=2026-01-01 10:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J39 | start=2026-01-03 02:00 | end=2026-01-03 06:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J58 | start=2026-01-03 11:00 | end=2026-01-03 21:00 | delay=1.0h
DEBUG:root:Serial SGS: scheduled J43 | start=2026-01-03 06:00 | end=2026-01-03 10:00 | delay=4.0h
DEBUG:root:Serial

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2']
waiting=['J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62']
candidates=['J2', 

,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,77.0,77.0,88.0,86.0,77.0,77.0,77.0,77.0,77.0,77.0,80.0,77.0,77.0,80.0,77.0,77.0,77.0,77.0,83.0


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,92.0,86.0,84.0,92.0,86.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,86,88,77,77,121,80,77,106,84,98,77,85,77,109,121


## Scheduling with 90 activities

In [5]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j90',
    file_name='j901_1.sm',
    json_path='j901_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=92 | CPM=69.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J2 | start=2026-01-01 01:00 | end=2026-01-01 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J15 | start=2026-01-01 11:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J20 | start=2026-01-01 18:00 | end=2026-01-01 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 09:00 | end=2026-01-01 17:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J12 | start=2026-01-01 17:00 | end=2026-01-01 19:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J5 | start=2026-01-01 02:00 | end=2026-01-01 04


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J48 | start=2026-01-02 07:00 | end=2026-01-02 16:00 | delay=5.0h
DEBUG:root:Serial SGS: scheduled J62 | start=2026-01-02 15:00 | end=2026-01-02 20:00 | delay=4.0h
DEBUG:root:Serial SGS: scheduled J74 | start=2026-01-02 17:00 | end=2026-01-03 02:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J42 | start=2026-01-02 08:00 | end=2026-01-02 16:00 | delay=8.0h
DEBUG:root:Serial SGS: scheduled J63 | start=2026-01-02 17:00 | end=2026-01-03 02:00 | delay=8.0h
DEBUG:root:Serial SGS: scheduled J66 | start=2026-01-02 09:00 | end=2026-01-02 17:00 | delay=5.0h
DEBUG:root:Serial SGS: scheduled J16 | start=2026-01-01 19:00 | end=2026-01-01 23:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J30 | start=2026-01-02 23:00 | end=2026-01-03 07:00 | delay=43.0h
DEBUG:root:Serial SGS: scheduled J39 | start=2026-01-01 15:00 | end=2026-01-01 20:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J19 | start=2026-01-02 16:00 | end=2026-01-03 01:00 | delay=38.0h
DEBUG:root:Serial 

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67', 'J68', 'J69', 'J70', 'J71', 'J72', 'J73', 'J74', 'J75', 'J76', 'J77', 'J78', 'J79', 'J80', 'J81', 'J82', 'J83', 'J84', 'J85', 'J86', 'J87', 'J88', 'J89', 'J90', 'J91', 'J92']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2']
waiting=['J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35

,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,82.0,83.0,98.0,94.0,87.0,84.0,89.0,80.0,81.0,77.0,88.0,99.0,99.0,88.0,83.0,90.0,81.0,80.0,80.0


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,209.0,86.0,89.0,97.0,94.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,94,98,83,82,111,88,89,101,95,108,84,101,84,102,148


## Scheduling with 120 activities

In [6]:
results_df_sgs, results_df_pgs, data_df = run_case(
    case_name='j120',
    file_name='j1201_1.sm',
    json_path='j1201_1.json',
    benchmark_data=benchmark_data
)

DEBUG:root:_precompute_availability_events: collected 2 boundary events
INFO:root:Starting Serial SGS | activities=122 | CPM=101.0h | rule=lf
DEBUG:root:Serial SGS: scheduled J1 | start=2026-01-01 00:00 | end=2026-01-01 01:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J3 | start=2026-01-01 01:00 | end=2026-01-01 05:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J6 | start=2026-01-01 05:00 | end=2026-01-01 08:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J7 | start=2026-01-01 08:00 | end=2026-01-01 18:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J11 | start=2026-01-01 18:00 | end=2026-01-02 00:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J4 | start=2026-01-01 01:00 | end=2026-01-01 03:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J13 | start=2026-01-01 03:00 | end=2026-01-01 10:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-02 00:00 | end=2026-01-02 09:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J8 | start=2026-01-01 03:00 | end=2026-01-01 0


OUTAGE DATA VALIDATION

✓ Schema validation passed
✓ All task IDs are unique
✓ All resource skill IDs are unique
✓ All equipment IDs are unique
✓ All location IDs are unique
✓ Task references checked
✓ Location references are valid
✓ Equipment references are valid
✓ Skill type references are valid
✓ Hold-point logic is valid
✓ No circular dependencies found

✓ VALIDATION PASSED


DEBUG:root:Serial SGS: scheduled J85 | start=2026-01-03 06:00 | end=2026-01-03 12:00 | delay=12.0h
DEBUG:root:Serial SGS: scheduled J101 | start=2026-01-02 18:00 | end=2026-01-03 00:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J56 | start=2026-01-02 11:00 | end=2026-01-02 21:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J78 | start=2026-01-03 06:00 | end=2026-01-03 11:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J96 | start=2026-01-03 00:00 | end=2026-01-03 01:00 | delay=18.0h
DEBUG:root:Serial SGS: scheduled J97 | start=2026-01-02 21:00 | end=2026-01-03 00:00 | delay=10.0h
DEBUG:root:Serial SGS: scheduled J18 | start=2026-01-03 01:00 | end=2026-01-03 10:00 | delay=11.0h
DEBUG:root:Serial SGS: scheduled J84 | start=2026-01-03 06:00 | end=2026-01-03 14:00 | delay=22.0h
DEBUG:root:Serial SGS: scheduled J99 | start=2026-01-03 01:00 | end=2026-01-03 06:00 | delay=0.0h
DEBUG:root:Serial SGS: scheduled J55 | start=2026-01-02 21:00 | end=2026-01-02 22:00 | delay=6.0h
DEBUG:root:Ser

t=2026-01-01 00:00
completed=[]
ongoing=['J1']
waiting=['J2', 'J3', 'J4', 'J5', 'J6', 'J7', 'J8', 'J9', 'J10', 'J11', 'J12', 'J13', 'J14', 'J15', 'J16', 'J17', 'J18', 'J19', 'J20', 'J21', 'J22', 'J23', 'J24', 'J25', 'J26', 'J27', 'J28', 'J29', 'J30', 'J31', 'J32', 'J33', 'J34', 'J35', 'J36', 'J37', 'J38', 'J39', 'J40', 'J41', 'J42', 'J43', 'J44', 'J45', 'J46', 'J47', 'J48', 'J49', 'J50', 'J51', 'J52', 'J53', 'J54', 'J55', 'J56', 'J57', 'J58', 'J59', 'J60', 'J61', 'J62', 'J63', 'J64', 'J65', 'J66', 'J67', 'J68', 'J69', 'J70', 'J71', 'J72', 'J73', 'J74', 'J75', 'J76', 'J77', 'J78', 'J79', 'J80', 'J81', 'J82', 'J83', 'J84', 'J85', 'J86', 'J87', 'J88', 'J89', 'J90', 'J91', 'J92', 'J93', 'J94', 'J95', 'J96', 'J97', 'J98', 'J99', 'J100', 'J101', 'J102', 'J103', 'J104', 'J105', 'J106', 'J107', 'J108', 'J109', 'J110', 'J111', 'J112', 'J113', 'J114', 'J115', 'J116', 'J117', 'J118', 'J119', 'J120', 'J121', 'J122']
candidates=['J1']
selected=['J1']
t=2026-01-01 01:00
completed=['J1']
ongoing=['J2

DEBUG:root:t=2026-01-04 08:00 | iter=69 | completed=84/122 | ongoing=3 | waiting=35 | candidates=3 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-04 09:00 | iter=70 | completed=85/122 | ongoing=3 | waiting=34 | candidates=3 | selected=1 | heap_size=10
DEBUG:root:t=2026-01-04 10:00 | iter=71 | completed=85/122 | ongoing=3 | waiting=34 | candidates=2 | selected=0 | heap_size=9
DEBUG:root:t=2026-01-04 12:00 | iter=72 | completed=86/122 | ongoing=3 | waiting=33 | candidates=3 | selected=1 | heap_size=9
DEBUG:root:t=2026-01-04 13:00 | iter=73 | completed=87/122 | ongoing=4 | waiting=31 | candidates=3 | selected=2 | heap_size=9
DEBUG:root:t=2026-01-04 15:00 | iter=74 | completed=88/122 | ongoing=4 | waiting=30 | candidates=3 | selected=1 | heap_size=8
DEBUG:root:t=2026-01-04 17:00 | iter=75 | completed=89/122 | ongoing=4 | waiting=29 | candidates=2 | selected=1 | heap_size=8
DEBUG:root:t=2026-01-04 18:00 | iter=76 | completed=89/122 | ongoing=4 | waiting=29 | candidates=1 | selected=0 | he

...

,lf,ls,ef,es,duration,random,mts,mtp,grpw,grd,rr,avgrr,maxrr,minrr,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,123.0,119.0,144.0,132.0,124.0,125.0,125.0,133.0,128.0,107.0,123.0,117.0,117.0,123.0,124.0,125.0,124.0,110.0,126.0


Results from LOGOS.CPM Using Parallel Generation Scheme:
------------------------------------------------------------


,first,max_use_res_ranked,max_use_res_shuffled,md_knapsack,look_ahead
0,310.0,120.0,148.0,148.0,120.0


Results from RCPSP
------------------------------------------------------------


,est,eft,lst,lft,spt,fifo,mts,rand,grpw,grd,mehh_8000_b,mehh_3375_b,mehh_1000_b,mehh_125_b,gphh_b
0,132,144,119,123,147,123,125,153,140,156,124,138,114,157,193
